In [45]:
import sys
from pathlib import Path

# Automatically locate AeroSentinel project root by searching upwards
current_path = Path.cwd().resolve()
PROJECT_ROOT = None
for parent in [current_path] + list(current_path.parents):
    if (parent / "data" / "dataset_inventory.yaml").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    # Fallback to standard directory structure
    PROJECT_ROOT = Path(r"C:\Users\Harsh\AeroSentinel").resolve()

AI_SERVICE_DIR = PROJECT_ROOT / "ai-service" if (PROJECT_ROOT / "ai-service").exists() else PROJECT_ROOT
if str(AI_SERVICE_DIR) not in sys.path:
    sys.path.insert(0, str(AI_SERVICE_DIR))

import yaml
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

from app.schemas.canonical import QualityFlag, DataStatus
from preprocessing.temporal_alignment import normalize_timestamp
from preprocessing.spatial_alignment import validate_coordinates, bind_h3_cell

print(f"Project root resolved: {PROJECT_ROOT}")
print(f"ai-service resolved:   {AI_SERVICE_DIR}")

Project root resolved: C:\Users\Harsh\AeroSentinel
ai-service resolved:   C:\Users\Harsh\AeroSentinel\ai-service


In [46]:
inventory_path = PROJECT_ROOT / "data" / "dataset_inventory.yaml"
assert inventory_path.exists(), f"Missing inventory file at {inventory_path}"

with open(inventory_path, "r", encoding="utf-8") as f:
    inventory = yaml.safe_load(f)

datasets = inventory.get("datasets", [])
print(f"Inventory Version: {inventory.get('version')}")
print(f"Total Registered Datasets: {len(datasets)}")

summary_rows = [
    {
        "ID": d.get("dataset_id") or d.get("id"),
        "Category": d.get("category"),
        "Name": d.get("dataset_name") or d.get("name"),
        "Spatial Res": d.get("spatial_resolution"),
        "Temporal Res": d.get("temporal_resolution"),
        "MVP Required": d.get("mvp_requirement") or d.get("required_for_mvp")
    }
    for d in datasets
]
pd.DataFrame(summary_rows)

Inventory Version: 1.1.0
Total Registered Datasets: 7


,ID,Category,Name,Spatial Res,Temporal Res,MVP Required
0,ds_air_cpcb_ground,AIR_QUALITY,CPCB / CAAQMS Continuous Ground Air Quality Ob...,Point monitoring stations (5 municipal station...,1 hour,REQUIRED
1,ds_weather_surface_obs,WEATHER,Surface Meteorological Observations & Reanalysis,0.1 degree grid (~11 km) anchored to Pune muni...,1 hour,REQUIRED
2,ds_fire_firms,FIRE,NASA FIRMS Thermal Anomalies & Active Fire Det...,375m (VIIRS I-band) / 1km (MODIS),Satellite overpass (1-2 times daily per sensor...,OPTIONAL_FOR_F0_F1_REQUIRED_PHASE_2
3,ds_satellite_s5p_tropomi,SATELLITE,Sentinel-5P TROPOMI Atmospheric Vertical Colum...,3.5km x 5.5km at nadir,Daily overpass (~13:30 local solar time),OPTIONAL_FOR_F0_F1_REQUIRED_PHASE_2
4,ds_citizen_hyperlocal_reports,CITIZEN,Citizen Hyperlocal Pollution Observations & Ph...,Mobile GPS coordinates (~5-15m accuracy),Event-driven (ad-hoc submission),OPTIONAL_FOR_F0_F1_REQUIRED_PHASE_4
5,ds_gis_reference_layers,GIS_REFERENCE,"Urban Infrastructure, Municipal Boundaries & R...",Sub-meter to 10m topological accuracy,Static snapshot (reviewed quarterly),REQUIRED
6,ds_demo_benchmark_pune,SYNTHETIC_DEMO,Master Aligned Ground Air & Surface Weather Fe...,Station coordinates bound to H3 Resolution 8 c...,1 hour (72-hour continuous time window),REQUIRED_FOR_TESTS


In [47]:
air_path = PROJECT_ROOT / "data" / "sample" / "air_quality.csv"
weather_path = PROJECT_ROOT / "data" / "sample" / "weather.csv"

air_df = pd.read_csv(air_path) if air_path.exists() else pd.DataFrame()
weather_df = pd.read_csv(weather_path) if weather_path.exists() else pd.DataFrame()

print(f"Air Quality raw shape: {air_df.shape}")
print(f"Weather raw shape:     {weather_df.shape}")

if not air_df.empty:
    print("\n--- Air Quality Missingness (%) ---")
    print(air_df.isna().mean().round(4) * 100)

if not weather_df.empty:
    print("\n--- Weather Missingness (%) ---")
    print(weather_df.isna().mean().round(4) * 100)

Air Quality raw shape: (10, 11)
Weather raw shape:     (10, 7)

--- Air Quality Missingness (%) ---
observed_at    0.0
station_id     0.0
latitude       0.0
longitude      0.0
pm25           0.0
pm2_5          0.0
pm10           0.0
no2            0.0
so2            0.0
co             0.0
o3             0.0
dtype: float64

--- Weather Missingness (%) ---
observed_at       0.0
station_id        0.0
temperature       0.0
humidity          0.0
wind_speed        0.0
wind_direction    0.0
pressure          0.0
dtype: float64


In [48]:
if not air_df.empty and "latitude" in air_df.columns and "longitude" in air_df.columns:
    sample_lat = air_df["latitude"].iloc[0]
    sample_lon = air_df["longitude"].iloc[0]
    is_valid, reason = validate_coordinates(sample_lat, sample_lon)
    h3_res8 = bind_h3_cell(sample_lat, sample_lon, resolution=8)
    print(f"Sample Point: ({sample_lat}, {sample_lon}) | Valid: {is_valid} ({reason}) | H3 Res 8: {h3_res8}")
else:
    print("Station coordinates resolved from station GeoJSON catalog during preprocessing.")

Sample Point: (19.076, 72.8777) | Valid: False (OUT_OF_BOUNDS_COORDINATES) | H3 Res 8: 88608b0b61fffff


In [49]:
if not air_df.empty and "pm25" in air_df.columns and len(air_df) > 3:
    pm_series = air_df["pm25"].dropna()
    
    # Target: T+1 (shift -1)
    y_true = pm_series.iloc[1:].values
    # Baseline Prediction: Value at T
    y_pred_persistence = pm_series.iloc[:-1].values
    # Historical Mean Baseline
    y_pred_mean = np.full_like(y_true, pm_series.mean())
    
    mae_persist = mean_absolute_error(y_true, y_pred_persistence)
    rmse_persist = np.sqrt(mean_squared_error(y_true, y_pred_persistence))
    
    mae_mean = mean_absolute_error(y_true, y_pred_mean)
    rmse_mean = np.sqrt(mean_squared_error(y_true, y_pred_mean))
    
    benchmark_df = pd.DataFrame([
        {"Model": "Naive Persistence (y_t)", "MAE (ug/m3)": round(mae_persist, 2), "RMSE (ug/m3)": round(rmse_persist, 2)},
        {"Model": "Historical Mean", "MAE (ug/m3)": round(mae_mean, 2), "RMSE (ug/m3)": round(rmse_mean, 2)}
    ])
    print("=== F0 Minimal Reference Baselines ===")
    print(benchmark_df.to_string(index=False))
else:
    print("Sample dataset size too small or missing pm25 column for statistical benchmark.")

=== F0 Minimal Reference Baselines ===
                  Model  MAE (ug/m3)  RMSE (ug/m3)
Naive Persistence (y_t)         6.61          7.93
        Historical Mean         8.14          9.55
